TEST COPY — targets are *_TEST / *_test only. Original files are never touched.

In [ ]:
# ================= 配置区域 (无需修改) =================
GITHUB_REPO_URL = "https://github.com/bdaconsulting1098-creator/bda_course.git"
COURSE_NAME = "Data_Course_101_TEST"  # 用户文件夹里显示的项目名
CATALOG_NAME = "workspace"       # 目标 Catalog
SCHEMA_NAME = "default"          # 目标 Schema
VOLUME_NAME = "course_data_test"      # 存放数据的 Volume

# 定义数据映射: { GitHub里的路径 : Volume里的子文件夹名 }
DATA_MAPPINGS = {
    "BDA1/data": "BDA1_Data",
    "BDA2/data": "BDA2_Data",
    "database_data": "Database_Data",
     "raw_data": "raw_Data",
      "Promotion_raw_data": "Promotion_raw_data"
}

# 定义需要复制给学生的课件文件夹 (排除 data)
COURSE_FOLDERS = ["BDA1", "BDA2"]

# ================= 自动执行逻辑 =================
import os
import shutil

# 1. 动态获取当前用户身份
current_user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
print(f"👋 欢迎, {current_user}! 正在为您准备课程环境...")

# 2. 定义路径
temp_clone_dir = f"/tmp/{COURSE_NAME}_temp"
user_workspace_dir = f"/Workspace/Users/{current_user}/{COURSE_NAME}"
volume_path = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}/"

# 3. 【新增功能】尝试自动创建 Volume
print(f"🛠️  正在检查 Volume 环境...")
try:
    # 使用 Spark SQL 创建 Volume (如果不存)
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.{VOLUME_NAME}")
    print(f"   ✅ Volume 已准备就绪: {CATALOG_NAME}.{SCHEMA_NAME}.{VOLUME_NAME}")
except Exception as e:
    print(f"   ⚠️ 自动创建 Volume 失败 (可能是权限不足)，尝试直接访问...")
    print(f"   错误信息: {str(e)}")

# 4. 清理旧环境 & 克隆仓库
print(f"⬇️  正在从 GitHub 克隆资料...")
if os.path.exists(temp_clone_dir):
    shutil.rmtree(temp_clone_dir, ignore_errors=True)

# 使用 magic command 运行 shell 命令
result = os.system(f"git clone {GITHUB_REPO_URL} {temp_clone_dir}")

if result != 0:
    raise Exception("❌ GitHub 克隆失败，请检查 URL 或网络连接。")

# 5. 部署数据到 Volume
print(f"📂 正在部署数据到 Volume: {volume_path} ...")
# 再次确保 OS 层面的目录存在
os.makedirs(volume_path, exist_ok=True)

for repo_subpath, volume_subfolder in DATA_MAPPINGS.items():
    src_path = os.path.join(temp_clone_dir, repo_subpath
                            )
    dst_path = os.path.join(volume_path, volume_subfolder)
    
    if os.path.exists(src_path):
        # 如果目标文件夹已存在，先删除旧的
        if os.path.exists(dst_path):
            shutil.rmtree(dst_path)
        # 复制整个文件夹到 Volume
        shutil.copytree(src_path, dst_path)
        print(f"   ✅ 数据已同步: {volume_subfolder}")
    else:
        print(f"   ⚠️ 警告: 在仓库中未找到 {repo_subpath}，跳过。")

# 6. 部署 Notebooks 到用户 Workspace
print(f"📘 正在部署课件到 Workspace: {user_workspace_dir} ...")

# 过滤函数：复制时忽略名为 'data'、'__pycache__' 及隐藏文件夹
def ignore_unwanted_folders(dir, files):
    ignore_list = []
    for f in files:
        if  f == '__pycache__' or f.startswith('.'):
            ignore_list.append(f)
    return ignore_list

for folder in COURSE_FOLDERS:
    src_path = os.path.join(temp_clone_dir, folder)
    dst_path = os.path.join(user_workspace_dir, folder)
    
    if os.path.exists(src_path):
        # if os.path.exists(dst_path):
        #     shutil.rmtree(dst_path)
        
        # 复制文件夹 (自动剔除 data、__pycache__、隐藏文件夹)
        shutil.copytree(src_path, dst_path, ignore=ignore_unwanted_folders, dirs_exist_ok=True)
        print(f"   ✅ 课件已安装: {folder}")
    else:
        print(f"   ⚠️ 未找到课件文件夹: {folder}")

# 7. 清理临时文件
shutil.rmtree(temp_clone_dir, ignore_errors=True)

print("="*40)
print(f"🚀 全部完成！")
print(f"1. 数据路径: {volume_path}")
print(f"2. 课件路径: {user_workspace_dir}")
print("👉 请刷新左侧 Workspace 查看您的文件。")

